# 🎬 Movie Ratings Analysis — Normalization & Standardization

In this notebook I'm going to:
1. Load a CSV of movie ratings by friends into a pandas DataFrame
2. Calculate average ratings per user and per movie
3. **Normalize** the ratings (min-max scaling, per user)
4. Discuss advantages and disadvantages of normalization
5. *(Extra credit)* **Standardize** the ratings (Z-score, per user)

The dataset has 6 friends rating up to 6 movies on a 1–5 scale. Not every friend rated every movie, so there are some missing values (NaN) we need to handle carefully.

---
## Step 0 — Import pandas

pandas is the go-to Python library for working with tabular data. It gives us the `DataFrame` object, which is basically a souped-up spreadsheet we can manipulate with code.

In [ ]:
import pandas as pd
from pathlib import Path

---
## Step 1 — Load the CSV into a DataFrame

A couple of things I had to figure out:
- `index_col=0` → tells pandas that the first column (the friends' names) is the **row label**, not actual data.
- `encoding='utf-8-sig'` → the file was probably saved from Excel, which sometimes sneaks a hidden BOM character (`\ufeff`) at the very start. This encoding strips it automatically so we don't get a weird column name.
- `Path.cwd()` → builds the file path relative to wherever this notebook is currently open. This avoids a `FileNotFoundError` when the notebook is opened from a different working directory. Just make sure the CSV file is in the **same folder** as this notebook!

In [ ]:
# Path.cwd() gives us the folder this notebook is currently running from.
# As long as the CSV sits in the same folder, this will always find it.
CSV_PATH = Path.cwd() / "Movie_Ratings_by_Friends.csv"

df = pd.read_csv(CSV_PATH, index_col=0, encoding="utf-8-sig")
print("Shape:", df.shape)   # (rows, columns)
df

---
## Step 2 — Average ratings (original data)

pandas' `.mean()` method automatically **skips NaN values** by default, which is exactly what we want here — we only average the movies a person actually rated.

- `axis=1` → average across columns → one number per **user** (row)
- `axis=0` → average across rows → one number per **movie** (column)

In [ ]:
print("--- Average rating per USER (original) ---")
print(df.mean(axis=1).round(2).to_frame(name="Avg Rating"))

print("\n--- Average rating per MOVIE (original) ---")
print(df.mean(axis=0).round(2).to_frame(name="Avg Rating"))

---
## Step 3 — Normalize ratings (min-max scaling, per user)

### What is normalization?

**Min-max normalization** rescales each value so that the user's **lowest rating becomes 0** and their **highest becomes 1**. Everything else lands proportionally in between.

$$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

### Why per user?

Some people are "harsh raters" (never go above 3) and others are "generous raters" (rarely go below 4). Normalizing per user removes that personal bias so we can fairly compare **relative preferences** across friends.

### Edge case

If a user only gave one rating, min == max, and we'd be dividing by zero. I handle that with a small `if` check — in that case, all their normalized scores are set to 0.5 (neutral).

In [ ]:
def normalize_row(row):
    """Apply min-max normalization to a single user's ratings."""
    row_min = row.min()   # pandas .min() ignores NaN automatically
    row_max = row.max()
    if row_max == row_min:   # guard against division by zero
        return row.apply(lambda x: 0.5 if pd.notna(x) else float("nan"))
    return (row - row_min) / (row_max - row_min)

# .apply(..., axis=1) calls our function once per ROW (once per user)
df_normalized = df.apply(normalize_row, axis=1)

print("NORMALIZED RATINGS (range 0–1 per user)")
df_normalized.round(4)

In [ ]:
print("--- Average rating per USER (normalized) ---")
print(df_normalized.mean(axis=1).round(4).to_frame(name="Avg Normalized"))

print("\n--- Average rating per MOVIE (normalized) ---")
print(df_normalized.mean(axis=0).round(4).to_frame(name="Avg Normalized"))

---
## Step 4 — Discussion: Normalized vs. Actual Ratings

### ✅ Advantages of normalized ratings

| Advantage | Why it matters |
|---|---|
| **Removes personal scale bias** | A "4" from a harsh rater is very different from a "4" from a generous one. Normalization puts everyone on equal footing. |
| **Highlights relative preferences** | We can now see which movies each user *relatively* liked most, even if their absolute scores were low. |
| **Consistent input range** | Recommendation engines and ML models often assume input values are in [0, 1]. Normalization makes this possible. |

### ❌ Disadvantages of normalized ratings

| Disadvantage | Why it matters |
|---|---|
| **Loses absolute information** | A normalized 1.0 could mean a 3 or a 5 — we can no longer tell if someone *actually* loved a movie. |
| **Sensitive to outliers** | One extreme rating (e.g., a single 1 or 5) shifts the entire scale for that user. |
| **Missing data complications** | Users with few ratings have less reliable min/max estimates, so their normalized values can be misleading. |
| **Harder to interpret averages** | A movie's average normalized score is not intuitive to explain to non-technical stakeholders. |

---
## Step 5 (Extra Credit) — Standardize ratings (Z-score, per user)

### What is standardization?

**Z-score standardization** rescales each user's ratings so their **mean becomes 0** and their **standard deviation becomes 1**:

$$
x_{\text{std}} = \frac{x - \bar{x}}{\sigma}
$$

A positive Z-score means the rating was **above that user's average**; negative means below.

### Normalization vs. Standardization — quick comparison

| | Normalization (min-max) | Standardization (Z-score) |
|---|---|---|
| Output range | Always **[0, 1]** | **Unbounded** (can be negative) |
| Centers data? | No | Yes (mean = 0) |
| Good for... | When you need a fixed range | When outliers are present / ML models that assume normal distribution |

### Edge case

If a user has only one rating (or all identical ratings), std = 0 → division by zero. I set those to 0.0 (which makes sense — they're all equally average).

In [ ]:
def standardize_row(row):
    """Apply Z-score standardization to a single user's ratings."""
    row_mean = row.mean()   # mean of non-NaN values
    row_std  = row.std()    # std of non-NaN values (ddof=1 by default)
    if pd.isna(row_std) or row_std == 0:
        return row.apply(lambda x: 0.0 if pd.notna(x) else float("nan"))
    return (row - row_mean) / row_std

df_standardized = df.apply(standardize_row, axis=1)

print("STANDARDIZED RATINGS (Z-score per user, mean=0, std=1)")
df_standardized.round(4)

In [ ]:
print("--- Average rating per USER (standardized) ---")
print(df_standardized.mean(axis=1).round(4).to_frame(name="Avg Z-score"))

print("\n--- Average rating per MOVIE (standardized) ---")
print(df_standardized.mean(axis=0).round(4).to_frame(name="Avg Z-score"))

### 💡 Key observation

Notice that every user's **average Z-score is exactly 0.0** — that's a mathematical guarantee of standardization! Each person's ratings are centered around their own mean.

The per-movie averages are now in Z-score units. A movie with a positive average (e.g., *Good Fortune*) was generally rated **above each person's personal average**, while a negative one (e.g., *Tron Ares*) was rated below. This is a much more nuanced signal than raw scores.

---
*Notebook by Julian — Information Systems Capstone, Spring 2026*